# PlantCare AI: genuine PlantVillage training and evaluation

This notebook executes the project's actual preparation, training and evaluation modules. It does not contain model weights, invented metrics or API credentials. Dataset results are separate from field-photo performance. The backend stays `model_unavailable` until an actual trusted artifact is imported.

1. On your PC, from `C:\Users\akhta\Downloads\PlantCareAi\backend`, run `python -m ml.package_training`.
2. In Colab choose **Runtime → Change runtime type → T4 GPU** (availability depends on your account).
3. Upload the generated `plantcare-training-code.zip` below.
4. Execute cells in order. Preparation needs several GB of disk and a network download. Training runtime varies; do not claim accuracy until the evaluation cell finishes.
5. Keep the downloaded bundle, reports and logs. No Gemini key is needed for training.


In [ ]:
from pathlib import Path
import platform, shutil, subprocess
ROOT = Path('/content/plantcare-training/backend')
ROOT.mkdir(parents=True, exist_ok=True)
print('Python:', platform.python_version())
print('Free disk GiB:', round(shutil.disk_usage('/content').free / 1024**3, 2))
subprocess.run(['nvidia-smi'], check=False)


In [ ]:
from google.colab import files
from zipfile import ZipFile
import io
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload exactly the plantcare-training-code.zip created by ml.package_training.')
archive_data = next(iter(uploaded.values()))
with ZipFile(io.BytesIO(archive_data)) as archive:
    members = archive.infolist()
    if len(members) > 50 or sum(item.file_size for item in members) > 2_000_000:
        raise ValueError('Unexpected archive size. Use the project training source packager.')
    for item in members:
        relative = Path(item.filename)
        allowed = (len(relative.parts) == 2 and relative.parts[0] == 'ml' and relative.suffix == '.py') or item.filename == 'requirements-training.txt'
        if relative.is_absolute() or '..' in relative.parts or not allowed:
            raise ValueError(f'Unexpected archive member: {item.filename}')
        target = ROOT / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(item))
print('Training source extracted to', ROOT)


## Install the pinned environment

The Linux TensorFlow CUDA extra supplies supported CUDA runtime packages. Colab's GPU driver must remain compatible. If Colab requests a runtime restart after installation, restart and rerun the setup cells; source/data files remain on the VM until the runtime is deleted. A fresh runtime with Python 3.12/3.13 is recommended. Native Windows TensorFlow 2.21 uses CPU; this notebook is the GPU path.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements-training.txt'), 'tensorflow[and-cuda]==2.21.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=True)


In [ ]:
import tensorflow as tf
print('TensorFlow:', tf.__version__, 'Keras:', tf.keras.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
if not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('No GPU detected. Choose a GPU runtime and resolve CUDA compatibility before the full training run.')
# Keep CPU data loading bounded; no decoded full-dataset cache is used.


## Inspect actual installed builder metadata, then download and prepare

PlantVillage TFDS 1.0.2 documents one original `train` split. The preparation command creates independent group-level train/validation/test manifests BEFORE augmentation. It records actual labels, filenames, class imbalance, invalid images, exact/perceptual/source groups and checksums. Source IDs are incomplete, so residual leakage is documented.

Mendeley version 1 declares CC0 1.0 for that dataset. TFDS uses its unaugmented subset excluding background; the TensorFlow documentation page license is a different license. See [TFDS](https://www.tensorflow.org/datasets/catalog/plant_village) and [source dataset](https://data.mendeley.com/datasets/tywbtsjrjv/1).


In [ ]:
import subprocess, sys

def run_module(module, *arguments):
    subprocess.run([sys.executable, '-m', module, *map(str, arguments)], cwd=ROOT, check=True)

run_module('ml.prepare_dataset', '--metadata-only')
run_module('ml.prepare_dataset', '--output', 'data/plant_village', '--seed', 42)


In [ ]:
import json
prepared = ROOT / 'data/plant_village'
dataset_metadata = json.loads((prepared / 'dataset_metadata.json').read_text())
print(json.dumps({key: dataset_metadata[key] for key in ['builder_version', 'valid_images', 'invalid_count', 'exact_duplicate_count', 'conflict_group_count', 'split_counts', 'class_split_counts']}, indent=2))


## Train the actual classifier

The first phase trains only a new head on ImageNet MobileNetV2; the second unfreezes the final convolution layers while keeping batch normalization fixed. RGB `[0,255]` inputs are resized by the shared Pillow function. A saved `Rescaling(1/127.5, offset=-1)` layer applies normalization exactly once. Validation determines checkpoints and early stopping. Test images never determine checkpoints or policy thresholds.

Use a new output version for every run. Full datasets and images are excluded from the deployment artifact.


In [ ]:
VERSION = 'pv-mobilenet-v2-v1'
run_module('ml.train_model', '--prepared', 'data/plant_village', '--output', f'models/{VERSION}', '--reports', f'reports/{VERSION}', '--model-version', VERSION, '--epochs', 15, '--fine-tune-epochs', 5, '--batch-size', 16, '--class-weight')


## Genuine evaluation and uncertainty policy

This cell evaluates held-out validation and test separately, saves per-class precision/recall/F1, ordered confusion matrices, ECE/Brier/NLL and actual score arrays. Score/margin policy thresholds are selected using validation only, then measured on test. They cannot prove unknown-species or non-leaf rejection. If validation evidence is inadequate the policy remains unvalidated and backend results remain uncertain.

Field-photo evaluation requires a separate permitted local manifest; see `ml/README.md`. Generated website assets are not valid field accuracy evidence.


In [ ]:
run_module('ml.evaluate_model', '--prepared', 'data/plant_village', '--bundle', f'models/{VERSION}', '--reports', f'reports/{VERSION}')
metadata = json.loads((ROOT / f'models/{VERSION}/model_metadata.json').read_text())
print(json.dumps(metadata['evaluation'], indent=2))
print(json.dumps(metadata['decision_policy'], indent=2))


## Export and import

The ZIP contains the actual model bundle and reports, not dataset images. Download before the Colab runtime disappears. After extracting this trusted ZIP on your PC, import the versioned model with the command printed below, then point backend environment paths at the new imported folder. A bundle checksum detects corruption; trust in the artifact source is still required.


In [ ]:
import zipfile
export = Path('/content') / f'plantcare-{VERSION}.zip'
with zipfile.ZipFile(export, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for folder in [ROOT / f'models/{VERSION}', ROOT / f'reports/{VERSION}']:
        for path in sorted(folder.rglob('*')):
            if path.is_file():
                archive.write(path, path.relative_to(ROOT))
print('Export:', export, 'bytes:', export.stat().st_size)
print(f'From backend on PC: python -m ml.import_model --source <extracted>/models/{VERSION} --output models/imported-{VERSION} --trust-local-artifact')
files.download(str(export))
